# 02 — Construção dos perfis dos baselines

Este notebook reproduz a etapa de construção dos perfis lexicais utilizados pelos baselines **POP** e **TF-IDF**.

O fluxo segue o notebook `perfis_autores_sbert.ipynb` e reutiliza a implementação consolidada em `src/preprocessing/build_profiles_baseline.py`.

São geradas duas representações:

1. `perfis_autores.json`, com os n-gramas candidatos agregados por pesquisador, preservando repetições para o cálculo de frequência nos baselines;
2. `perfis_por_documento.json`, com os n-gramas separados por publicação, utilizado posteriormente no cálculo de Coverage.

O processamento mantém as decisões do experimento: concatenação de título, palavras-chave e resumo, detecção de idioma por sentença, normalização lexical, modelos spaCy em português e inglês, n-gramas de 1 a 3 termos e filtragem morfossintática por POS.

## 1. Dependências

As bibliotecas e os modelos linguísticos abaixo são necessários para reproduzir o processamento dos perfis dos baselines.

In [ ]:
%pip install -q spacy langdetect tqdm

import spacy.util
import subprocess
import sys

for model_name in ("pt_core_news_sm", "en_core_web_sm"):
    if not spacy.util.is_package(model_name):
        subprocess.run(
            [sys.executable, "-m", "spacy", "download", model_name],
            check=True,
        )

## 2. Configuração do projeto

A raiz do repositório é localizada automaticamente pela presença da pasta `src`.

In [ ]:
from pathlib import Path
from multiprocessing import cpu_count
import json
import sys

current = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [current, *current.parents] if (p / "src").exists()), current)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing.build_profiles_baseline import (
    BATCH_SPACY,
    carregar_autores_qrels,
    gerar_perfis_autores,
    gerar_perfis_documentos,
)

print(f"Raiz do projeto: {PROJECT_ROOT}")

## 3. Caminhos de entrada e saída

Informe apenas o diretório que contém os arquivos processados da coleção LExR. Os nomes seguem os utilizados no experimento.

In [ ]:
SOURCE_DIR = Path("CAMINHO_PARA_OS_ARQUIVOS_PROCESSADOS")

DOCUMENTS = SOURCE_DIR / "filtered_documents.json"
QRELS = SOURCE_DIR / "LExR-prof-qrels_filtrado"

OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "baseline"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

AUTHOR_PROFILES = OUTPUT_DIR / "perfis_autores.json"
DOCUMENT_PROFILES = OUTPUT_DIR / "perfis_por_documento.json"

for name, path in {
    "filtered_documents.json": DOCUMENTS,
    "LExR-prof-qrels_filtrado": QRELS,
}.items():
    print(f"{name:35s} -> {'OK' if path.exists() else 'não encontrado'}")

## 4. Parâmetros do processamento

Os valores abaixo reproduzem as configurações do notebook. As repetições dos n-gramas são preservadas, pois POP e TF-IDF dependem das frequências observadas no perfil de cada pesquisador.

In [ ]:
N_WORKERS = max(2, cpu_count() - 1)
BATCH_SIZE_SPACY = BATCH_SPACY

print(f"Workers: {N_WORKERS}")
print(f"Batch spaCy: {BATCH_SIZE_SPACY}")

## 5. Universo de autores

Os pesquisadores considerados são definidos pela primeira coluna do arquivo `LExR-prof-qrels_filtrado`.

In [ ]:
autores_alvo = carregar_autores_qrels(QRELS)
print(f"Autores no ground truth filtrado: {len(autores_alvo):,}")

## 6. Construção do perfil agregado por autor

As publicações de cada pesquisador são concatenadas seguindo a ordem **título → palavras-chave → resumo**. O texto é segmentado antes da normalização para impedir que n-gramas atravessem fronteiras textuais.

Depois são aplicados detecção de idioma por sentença, normalização lexical, POS tagging e extração de n-gramas de 1 a 3 termos. O resultado mantém todas as ocorrências repetidas.

In [ ]:
gerar_perfis_autores(
    caminho_documentos=DOCUMENTS,
    autores_alvo=autores_alvo,
    caminho_saida=AUTHOR_PROFILES,
    workers=N_WORKERS,
    batch_spacy=BATCH_SIZE_SPACY,
)

## 7. Construção dos perfis por documento

A mesma pipeline linguística é aplicada separadamente a cada publicação. Essa saída preserva o vínculo entre documento e n-gramas e é utilizada posteriormente no cálculo da métrica Coverage.

In [ ]:
gerar_perfis_documentos(
    caminho_documentos=DOCUMENTS,
    autores_alvo=autores_alvo,
    caminho_saida=DOCUMENT_PROFILES,
    workers=N_WORKERS,
    batch_spacy=BATCH_SIZE_SPACY,
)

## 8. Verificação das saídas

A conferência abaixo apresenta apenas estatísticas e pequenas amostras dos arquivos gerados.

In [ ]:
with AUTHOR_PROFILES.open("r", encoding="utf-8") as f:
    perfis_autores = json.load(f)

with DOCUMENT_PROFILES.open("r", encoding="utf-8") as f:
    perfis_documentos = json.load(f)

n_tags = sum(len(tags) for tags in perfis_autores.values())
n_docs = sum(len(docs) for docs in perfis_documentos.values())

print(f"Autores em perfis_autores.json: {len(perfis_autores):,}")
print(f"Ocorrências de n-gramas: {n_tags:,}")
print(f"Autores em perfis_por_documento.json: {len(perfis_documentos):,}")
print(f"Vínculos autor-documento: {n_docs:,}")

if perfis_autores:
    author_id = next(iter(perfis_autores))
    print("\nExemplo de autor:", author_id)
    print("Primeiras candidatas:", perfis_autores[author_id][:20])

if perfis_documentos:
    author_id = next(iter(perfis_documentos))
    docs = perfis_documentos[author_id]
    if docs:
        doc_id = next(iter(docs))
        print("\nExemplo por documento:")
        print("Autor:", author_id)
        print("Documento:", doc_id)
        print("Primeiras candidatas:", docs[doc_id][:20])

## 9. Arquivos produzidos

Ao final da execução, este notebook gera:

- `perfis_autores.json`, entrada comum dos modelos POP e TF-IDF;
- `perfis_por_documento.json`, representação documental empregada na avaliação de Coverage.

O ranqueamento das candidatas não é realizado neste notebook. Essa etapa pertence aos experimentos específicos dos baselines.